In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

pd.set_option('display.max_columns', 100)
df = pd.read_csv("AmesHousing.csv", sep = '\t')


display(df.head())
display(df.describe())

#all the columns
display(df.columns)
#the shape of the data frame
display(df.shape)

#the type of each column
display (df.dtypes)

#missing values per column
missing_counts = df.isnull().sum()
display(missing_counts[missing_counts > 0].sort_values(ascending=False))





In [ ]:
# Addressing missing values

df_clean = pd.read_csv(
    "AmesHousing.csv", sep = '\t',
    keep_default_na=False,  # pandas turns the strings "NA"/"None" into NaN; here they mean the feature is absent (no pool, no alley), not missing data
    na_values=[""],
)

numeric_cols = ["Lot Frontage", "Mas Vnr Area", "Garage Yr Blt", "BsmtFin SF 1", "BsmtFin SF 2",
                "Bsmt Unf SF", "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath",
                "Garage Area", "Garage Cars"]
df_clean[numeric_cols] = df_clean[numeric_cols].apply(pd.to_numeric, errors="coerce") # Convert to numeric, setting errors to NaN for non-numeric values

absence_cols = ["Pool QC", "Misc Feature", "Alley", "Fence", "Mas Vnr Type", "Fireplace Qu",
                "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
                "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]
df_clean[absence_cols] = df_clean[absence_cols].replace("NA", "None").fillna("None") # Replace "NA" with "None" and fill NaN values with "None"

# 1 if the house has a pool, 0 if not.
df_clean["Has Pool"] = (df_clean["Pool QC"] != "None").astype(int)
# 1 if the house has a garage, 0 if not.
df_clean["Has Garage"] = (df_clean["Garage Type"] != "None").astype(int)
# 1 if the house has a basement, 0 if not.
df_clean["Has Basement"] = (df_clean["Bsmt Qual"] != "None").astype(int)
# 1 if the house has a fireplace, 0 if not.
df_clean["Has Fireplace"] = (df_clean["Fireplace Qu"] != "None").astype(int)
# 1 if the house has a fence, 0 if not.
df_clean["Has Fence"] = (df_clean["Fence"] != "None").astype(int)
# 1 if the house has an alley, 0 if not.
df_clean["Has Alley"] = (df_clean["Alley"] != "None").astype(int)
# 1 if the house has an extra feature (shed, tennis court...), 0 if not.
df_clean["Has Misc Feature"] = (df_clean["Misc Feature"] != "None").astype(int)
# 1 where Lot Frontage is empty. Must run before it gets filled below.
df_clean["Lot Frontage Missing"] = df_clean["Lot Frontage"].isnull().astype(int)

# Numeric columns that are empty only when the feature does not exist.
zero_cols = ["Garage Area", "Garage Cars", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF",
             "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath", "Mas Vnr Area"]
# No garage or basement means size 0, so empty becomes 0.
df_clean[zero_cols] = df_clean[zero_cols].fillna(0)

# Empty lot frontage gets the median of houses in the same neighborhood.
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(df_clean.groupby("Neighborhood")["Lot Frontage"].transform("median"))
# If a whole neighborhood is empty, use the median of all houses.
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(df_clean["Lot Frontage"].median())

# No garage means no garage year. The year the house was built is used instead.
df_clean["Garage Yr Blt"] = df_clean["Garage Yr Blt"].fillna(df_clean["Year Built"])
# One house has no electrical system recorded. It gets the most common type.
df_clean["Electrical"] = df_clean["Electrical"].fillna(df_clean["Electrical"].mode()[0])

# Counts the cells still empty. The result should be 0.
print(df_clean.isnull().sum().sum())

display(df_clean.head())
display(df_clean.describe())

#missing values per column
missing_counts = df_clean.isnull().sum()
display(missing_counts[missing_counts > 0].sort_values(ascending=False))

In [ ]:
#Finding possible outliers using IQR
num_cols = [c for c in df.select_dtypes("number").columns
            if df[c].nunique() >= 20 and c not in ("Order", "PID")]

flags = pd.DataFrame(False, index=df.index, columns=num_cols)
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    flags[c] = (df[c] < q1 - 3 * iqr) | (df[c] > q3 + 3 * iqr)

candidates = pd.DataFrame({
    "n_flags": flags.sum(axis=1),
    "flagged_columns": flags.apply(lambda r: ", ".join(r.index[r]), axis=1),
})
candidates = candidates[candidates["n_flags"] > 0].sort_values("n_flags", ascending=False)

#columns have the most extreme values
display(flags.sum().sort_values(ascending=False).to_frame("n_outliers"))

#top 30 candidate rows
display(candidates.head(30))

#key values for the top 30 candidates
display(df.loc[candidates.index[:30], ["Gr Liv Area", "Lot Area", "Total Bsmt SF", "SalePrice"]])

In [ ]:
# 1. Isolate the numerical columns
numeric_df = df.select_dtypes(include=['number'])

# 2. Calculate the correlation matrix
# This computes the Pearson correlation coefficient between all pairs of numeric columns
corr_matrix = numeric_df.corr()

# 3. Visualize with a heatmap
plt.figure(figsize=(12, 10))  # Increase figure size so it is readable
sns.heatmap(corr_matrix, 
            annot=False,      # Set to True if you want the exact numbers (gets messy with many columns)
            cmap='coolwarm',  # Red = positive correlation, Blue = negative correlation
            fmt=".2f")        
plt.title("Correlation Matrix of Ames Housing Features")
plt.show()

In [ ]:
df.shape

In [ ]:
sns.violinplot(data=df, x="Lot Shape", y="SalePrice", inner="quartille", cut = 0)
plt.xlabel("Lot Shape")
plt.ylabel("SalePrice")
plt.title("Price distribution by Lot Shape")
plt.tight_layout()
plt.show()

In [ ]:
sns.violinplot(data=df, x="Condition 1", y="SalePrice", inner="quartille", cut = 0)
plt.xlabel("Condition 1")
plt.ylabel("SalePrice")
plt.title("Price distribution by Condition 1")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("Overall Qual")["SalePrice"].median().plot(kind="bar")
plt.xlabel("Overall Quality")
plt.ylabel("Median Sale Price")
plt.title("Median Sale Price by Overall Quality")
plt.tight_layout()
plt.show()



In [ ]:
median_price_by_year = df.groupby("Yr Sold")["SalePrice"].median()

median_price_by_year.plot(kind="line", marker="o")
plt.xlabel("Year Sold")
plt.ylabel("Median Sale Price")
plt.title("Median Sale Price Over Time")
plt.xticks(median_price_by_year.index)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Gr Liv Area", y="SalePrice", alpha=0.6)

plt.xlabel("Above-ground Living Area (sq ft)")
plt.ylabel("Sale Price ($)")
plt.title("Sale Price vs. Living Area")
plt.tight_layout()
plt.show()

In [ ]:
house_counts = df["House Style"].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    house_counts,
    labels=house_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"width": 0.4}
)

ax.set_title("Share of Homes by House Style")
ax.axis("equal")
plt.tight_layout()
plt.show()

In [ ]:
counts = df["House Style"].value_counts()

#changing the 0.05 includes more house styles in the pie chart making the "Other" category smaller (small means a list of house styles in "Other")
small = counts[counts / counts.sum() < 0.05].index
plot_counts = counts.drop(small)
if len(small):
    plot_counts = pd.concat([plot_counts, pd.Series({"Other": counts[small].sum()})])

colors = plt.cm.Set1(range(len(plot_counts)))
fig, ax = plt.subplots(figsize=(9, 6))
wedges, _, _ = ax.pie(
    plot_counts,
    colors=colors,
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.1f}%" if p >= 4 else "",
    pctdistance=0.8,
    wedgeprops={"width": 0.4, "edgecolor": "white"},
)
ax.text(0, 0, f"{counts.sum():,}\nhomes", ha="center", va="center", fontsize=14, weight="bold")

labels = [f"{n} ({c:,})" for n, c in plot_counts.items()]
ax.legend(wedges, labels, title="House Style", loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
ax.set_title("Share of Homes by House Style", fontsize=14, weight="bold")
ax.axis("equal")
plt.tight_layout()
plt.show()

display(small)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=df,
    x="House Style",
    y="SalePrice",
    estimator="mean",
    errorbar=("ci", 95),
    capsize=0.1
)

plt.xlabel("House Style")
plt.ylabel("Mean Sale Price")
plt.title("Mean Sale Price by House Style (95% CI)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# SalePrice Histogram
plt.figure(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True, bins=40, color='teal')
plt.title('SalePrice Distribution')
plt.xlabel('SalePrice ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Living Area Histogram
plt.figure(figsize=(10, 5))
sns.histplot(df['Gr Liv Area'], kde=True, bins=40, color='purple')
plt.title('Gr Liv Area Distribution')
plt.xlabel('Above-ground Living Area (sq ft)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# SalePrice vs. Overall Quality Box Plot
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Overall Qual', y='SalePrice', hue='Overall Qual', palette='Set3', legend=False)
plt.title('Box Plot: Sale Price vs. Overall Quality')
plt.xlabel('General Quality')
plt.ylabel('SalePrice')
plt.tight_layout()
plt.show()

In [ ]:
df = df[['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 
       #'Alley', 
       'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 
       #'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 
       #'Fireplace Qu', 
       'Garage Type', 'Garage Yr Blt',
       'Garage Finish', 
       'Garage Cars', 
       # 'Garage Area', 
       'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Wood Deck SF', 'Open Porch SF',
       'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 
       #'Pool QC','Fence', 'Misc Feature', 'Yr Sold',
       'Misc Val', 'Mo Sold', 'Sale Type',
       'Sale Condition', 'SalePrice']].copy()